# CasCrop: Asymmetric Cascade Diffusion for Crop Waste Prediction
**Full experiments for Nature Communications submission.**

Architecture: pre-computed multi-hop asymmetric shock diffusion + vulnerability gating + exposure attention.  
4 ablation models × 5 seeds × 100 epochs on 638K samples.  
~30 min on T4 GPU (no online GNN — standard mini-batch training).

In [ ]:
#@title Configuration
QUICK_TEST = True  #@param {type:"boolean"}
SEEDS = [42, 123, 456] if QUICK_TEST else [42, 123, 456, 789, 1024]
EPOCHS = 30 if QUICK_TEST else 100
PATIENCE = 10 if QUICK_TEST else 20
BATCH_SIZE = 2048 if QUICK_TEST else 1024
print(f"{'QUICK' if QUICK_TEST else 'FULL'}: {len(SEEDS)} seeds, {EPOCHS} epochs")

In [ ]:
#@title Setup
import torch, os, sys, json, time, shutil, subprocess
import numpy as np, pandas as pd
from pathlib import Path

if torch.cuda.is_available():
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {torch.cuda.get_device_name(0)} ({gpu_mem:.1f}GB)')
    if gpu_mem < 8: BATCH_SIZE = min(BATCH_SIZE, 512)
else:
    print('NO GPU - Runtime > Change runtime type > T4')

if not os.path.exists('CasCrop'):
    !git clone https://github.com/keshavkrishnan08/CasCrop.git
if os.path.basename(os.getcwd()) != 'CasCrop':
    os.chdir('CasCrop')
!pip install -q pandas pyarrow scipy scikit-learn statsmodels seaborn tqdm pyyaml 2>&1 | tail -1
sys.path.insert(0, 'src')
for d in ['checkpoints','results','paper/figures','paper/tables']:
    os.makedirs(d, exist_ok=True)

SAVE_TO_DRIVE = False; DRIVE_PATH = ''
try:
    from google.colab import drive; drive.mount('/content/drive')
    DRIVE_PATH = '/content/drive/MyDrive/CasCrop_Results'
    os.makedirs(f'{DRIVE_PATH}/checkpoints', exist_ok=True)
    for f in Path(f'{DRIVE_PATH}/checkpoints').glob('*.pt'):
        if not Path(f'checkpoints/{f.name}').exists(): shutil.copy2(f, f'checkpoints/{f.name}')
    SAVE_TO_DRIVE = True; print(f'Drive: {DRIVE_PATH}')
except: print('No Drive')

def backup():
    if not SAVE_TO_DRIVE: return
    for d in ['results','checkpoints','paper/figures','paper/tables']:
        if not os.path.exists(d): continue
        dst = f'{DRIVE_PATH}/{d}'; os.makedirs(dst, exist_ok=True)
        for f in Path(d).glob('*'):
            if f.is_file(): shutil.copy2(f, f'{dst}/{f.name}')
print('OK')

In [ ]:
#@title Load Data + Pre-compute Diffusion (~10 sec)
if Path('data/processed/features_monthly.parquet').exists() and Path('data/graphs/combined_graph.npz').exists():
    print('Raw data present.')
else:
    !wget -q --show-progress -O monthly.tar.gz https://github.com/keshavkrishnan08/CasCrop/releases/download/v0.1-data/cascrop_monthly.tar.gz
    !tar xzf monthly.tar.gz && rm monthly.tar.gz

assert Path('data/processed/features_monthly.parquet').exists(), 'Monthly features missing!'
assert Path('data/graphs/combined_graph.npz').exists(), 'Graph missing!'

if not Path('data/processed/features_diffusion.parquet').exists():
    print('Pre-computing asymmetric cascade diffusion features...')
    !python scripts/precompute_diffusion.py
else:
    print('Diffusion features already computed.')

assert Path('data/processed/features_diffusion.parquet').exists(), 'Diffusion pre-computation failed!'

f = pd.read_parquet('data/processed/features_diffusion.parquet')
l = pd.read_parquet('data/processed/labels_monthly.parquet')
with open('data/processed/feature_groups_diffusion.json') as fj: groups = json.load(fj)
print(f'{len(f):,} samples | {f["fips"].nunique()} counties | {l["waste"].mean():.1%} waste')
print(f'bio={len(groups["biophysical"])}, econ={len(groups["economic"])}, hist={len(groups["historical"])}, diff={len(groups["diffusion"])}')

In [ ]:
#@title Smoke Test
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
from models.cascade_net import CascadeNet
_N = 64
_batch = {
    'x_bio': torch.randn(_N, len(groups['biophysical']), device=device),
    'x_econ': torch.randn(_N, len(groups['economic']), device=device),
    'x_hist': torch.randn(_N, len(groups['historical']), device=device),
    'x_diff': torch.randn(_N, len(groups['diffusion']), device=device),
}
model = CascadeNet(
    bio_dim=len(groups['biophysical']), econ_dim=len(groups['economic']),
    hist_dim=len(groups['historical']), diff_dim=len(groups['diffusion']),
).to(device)
out = model(_batch)
out['waste_logits'].sum().backward()
print(f'CascadeNet: {sum(p.numel() for p in model.parameters()):,} params, device={device}')
print(f'Diffusion attention shape: {out["diffusion_attention"].shape}')
del model; torch.cuda.empty_cache() if torch.cuda.is_available() else None
print('Smoke test passed')

---
## Experiment 1: Cascade Ablation
| Row | Model | Features | Graph |
|-----|-------|----------|-------|
| 1 | Local Only | Bio + Hist | No |
| 2 | Local + Econ | Bio + Econ + Hist | No |
| 3 | Symmetric Diffusion | All + symmetric hops | Yes (symmetric) |
| 4 | **CasCrop (ours)** | All + asymmetric hops + attention | Yes (asymmetric) |

In [ ]:
%%time
models_str = 'local_only local_econ symmetric_diff cascade_net'
ss = ' '.join(str(s) for s in SEEDS)

print(f'Training 4 ablation models x {len(SEEDS)} seeds x {EPOCHS} epochs...')
r = subprocess.run(
    f'python scripts/train_cascade.py --models {models_str} --seeds {ss} '
    f'--epochs {EPOCHS} --patience {PATIENCE} --batch-size {BATCH_SIZE} --gpu 0',
    shell=True, capture_output=True, text=True, timeout=14400)
for line in r.stdout.strip().split('\n')[-30:]: print(line)
if r.returncode != 0:
    print(f'EXIT CODE: {r.returncode}')
    if r.stderr: print(f'STDERR:\n{r.stderr[-1000:]}')
backup()
if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
# Results table
models = ['local_only', 'local_econ', 'symmetric_diff', 'cascade_net']
if os.path.exists('results/cascade_results.json'):
    with open('results/cascade_results.json') as f: res = json.load(f)
    df = pd.DataFrame(res)
    print(f'{"Model":<20} {"AUC-ROC":>14} {"F1":>8} {"AP":>8}')
    print('-'*55)
    for m in models:
        d = df[(df['model']==m) & (df['test_auc_roc']>0)]
        if len(d):
            print(f'{m:<20} {d["test_auc_roc"].mean():.3f}+/-{d["test_auc_roc"].std():.3f}'
                  f'  {d["test_f1"].mean():.3f}  {d["test_auc_pr"].mean():.3f}')
    print()
    for base in ['local_only', 'local_econ', 'symmetric_diff']:
        c = df[df['model']=='cascade_net']['test_auc_roc'].mean()
        b = df[df['model']==base]['test_auc_roc'].mean()
        print(f'CasCrop vs {base:<18} DAUC={c-b:+.4f}')
else:
    print('No results yet')

---
## Statistical Tests

In [ ]:
if os.path.exists('results/cascade_results.json'):
    with open('results/cascade_results.json') as f: res = json.load(f)
    df = pd.DataFrame(res)
    from evaluation.statistical_tests import paired_ttest_across_seeds
    ca = sorted(df[df['model']=='cascade_net']['test_auc_roc'].tolist())
    if len(ca) >= 2:
        print(f'{"Comparison":<40} {"DAUC":>7} {"p":>8} {"Sig":>5}')
        print('-'*63)
        for m in ['local_only','local_econ','symmetric_diff']:
            ma = sorted(df[df['model']==m]['test_auc_roc'].tolist())
            if len(ma) != len(ca): continue
            t = paired_ttest_across_seeds(ca, ma)
            sig = '***' if t['p_value']<.001 else '**' if t['p_value']<.01 else '*' if t['p_value']<.05 else 'n.s.'
            print(f'CasCrop vs {m:<28} {t["mean_diff"]:>+.4f} {t["p_value"]:>8.4f} {sig:>5}')
else:
    print('No results yet')

---
## Figure: Ablation Bar Chart

In [ ]:
import matplotlib.pyplot as plt
import matplotlib; matplotlib.rcParams.update({'font.size':9,'figure.dpi':300})

if 'df' in dir() and len(df) > 0:
    mo = ['local_only','local_econ','symmetric_diff','cascade_net']
    dn = ['R1: Local\n(Bio only)','R2: Local+Econ\n(No graph)','R3: Symmetric\n(Sym diffusion)','R4: CasCrop\n(Asym+Attention)']
    co = ['#7f8c8d','#3498db','#9b59b6','#e74c3c']
    ms = [df[df['model']==m]['test_auc_roc'].mean() for m in mo]
    ss_ = [df[df['model']==m]['test_auc_roc'].std() if len(df[df['model']==m])>1 else 0 for m in mo]
    
    fig, ax = plt.subplots(figsize=(7,3.5))
    bars = ax.bar(range(4), ms, 0.6, yerr=ss_, capsize=4, color=co, edgecolor='k', linewidth=.5)
    for b, v in zip(bars, ms):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.008, f'{v:.3f}', ha='center', fontsize=7)
    ax.set_xticks(range(4)); ax.set_xticklabels(dn, fontsize=7)
    ax.set_ylim(0.7, 1.0); ax.set_ylabel('AUC-ROC'); ax.grid(axis='y', alpha=0.3)
    ax.set_title('Cascade Diffusion Ablation (638K Monthly Samples)', fontweight='bold')
    plt.tight_layout()
    fig.savefig('paper/figures/fig3_cascade_ablation.pdf', dpi=300, bbox_inches='tight')
    plt.show()
    print('Saved: paper/figures/fig3_cascade_ablation.pdf')

---
## Experiment 2: Graph Perturbation

In [ ]:
%%time
import shutil

# Back up originals
shutil.copy('data/graphs/combined_graph.npz', 'data/graphs/combined_backup.npz')
shutil.copy('data/processed/features_diffusion.parquet', 'data/processed/features_diffusion_backup.parquet')
if os.path.exists('results/cascade_results.json'):
    shutil.copy('results/cascade_results.json', 'results/cascade_results_main.json')
for f in Path('checkpoints').glob('cascade_cascade_net_*.pt'):
    shutil.copy(f, f'{f}.main_bak')

# Shuffled graph
g = np.load('data/graphs/combined_backup.npz'); np.random.seed(42)
np.savez('data/graphs/combined_graph.npz',
         edge_index=np.array([g['edge_index'][0], np.random.permutation(g['edge_index'][1])]),
         edge_weight=g['edge_weight'])

# Re-compute diffusion on shuffled graph
os.remove('data/processed/features_diffusion.parquet')
!python scripts/precompute_diffusion.py 2>&1 | tail -3

# Train on shuffled
ss3 = ' '.join(str(s) for s in SEEDS[:3])
r = subprocess.run(
    f'python scripts/train_cascade.py --models cascade_net --seeds {ss3} '
    f'--epochs {EPOCHS} --patience {PATIENCE} --batch-size {BATCH_SIZE} --gpu 0',
    shell=True, capture_output=True, text=True, timeout=3600)
for line in r.stdout.strip().split('\n')[-10:]: print(line)

# Save perturbation results
if os.path.exists('results/cascade_results.json'):
    shutil.copy('results/cascade_results.json', 'results/perturbation_results.json')

# Restore originals
shutil.copy('data/graphs/combined_backup.npz', 'data/graphs/combined_graph.npz')
shutil.copy('data/processed/features_diffusion_backup.parquet', 'data/processed/features_diffusion.parquet')
for f in Path('checkpoints').glob('cascade_cascade_net_*.pt.main_bak'):
    shutil.copy(f, str(f).replace('.main_bak', '')); f.unlink()
if os.path.exists('results/cascade_results_main.json'):
    shutil.copy('results/cascade_results_main.json', 'results/cascade_results.json')

# Compare
if os.path.exists('results/perturbation_results.json'):
    with open('results/perturbation_results.json') as fp: pert = json.load(fp)
    with open('results/cascade_results.json') as fm: main = json.load(fm)
    pert_auc = pd.DataFrame(pert).query('model=="cascade_net"')['test_auc_roc'].mean()
    main_auc = pd.DataFrame(main).query('model=="cascade_net"')['test_auc_roc'].mean()
    print(f'\nPerturbation Analysis:')
    print(f'  Real graph AUC:     {main_auc:.4f}')
    print(f'  Shuffled graph AUC: {pert_auc:.4f}')
    print(f'  Delta:              {main_auc - pert_auc:+.4f}')
    print(f'  Graph signal: {"CONFIRMED" if main_auc > pert_auc else "NOT CONFIRMED"}')

if torch.cuda.is_available(): torch.cuda.empty_cache()

---
## Download

In [ ]:
backup()
!tar czf /content/cascrop_cascade_results.tar.gz results/ paper/figures/ paper/tables/ checkpoints/
try:
    from google.colab import files; files.download('/content/cascrop_cascade_results.tar.gz')
except: print('Download from Files or Drive')
print('DONE')